In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

# STEP 1: Dataset Path
DATASET_PATH = "/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VAL_PATH = os.path.join(DATASET_PATH, "valid")

print("Train path:", TRAIN_PATH)
print("Validation path:", VAL_PATH)
print("Train exists:", os.path.exists(TRAIN_PATH))
print("Validation exists:", os.path.exists(VAL_PATH))

# STEP 2: Image Parameters
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# STEP 3: Load Training Dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

# STEP 4: Load Validation Dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

# STEP 5: Classes
class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:")
print(class_names)
print("Number of classes:", num_classes)

# STEP 6: Normalize Pixel Values
normalization = tf.keras.layers.Rescaling(1.0 / 255)

# STEP 7: Improve Dataset Performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# STEP 8: Display Sample Images
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.show()

# STEP 9: CNN Model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    normalization,
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

# STEP 10: Compile Model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# STEP 11: Model Summary
model.summary()

# STEP 12: Train Model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

# STEP 13: Evaluate Model
loss, accuracy = model.evaluate(val_ds)
print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)
print("Validation Accuracy (%):", accuracy * 100)

# STEP 14: Accuracy Graph
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training and Validation Accuracy")
plt.legend()
plt.show()

# STEP 15: Loss Graph
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Training and Validation Loss")
plt.legend()
plt.show()

# STEP 16: Save Model
MODEL_PATH = "/kaggle/working/plant_disease_cnn.keras"
model.save(MODEL_PATH)
print("Model saved successfully!")
print("Saved at:", MODEL_PATH)

In [ ]:
image_path = "/kaggle/input/datasets/akashsaha/plat-image/Potato-leaf-blight.webp"

In [ ]:
# Load test image
img = tf.keras.utils.load_img(
    image_path,
    target_size=IMG_SIZE
)

# Convert image to array
img_array = tf.keras.utils.img_to_array(img)

# Create batch
img_array = tf.expand_dims(img_array, 0)

# Make prediction
predictions = model.predict(img_array)

# Model already uses softmax
score = predictions[0]

# Get predicted class
predicted_class = class_names[np.argmax(score)]

# Get confidence
confidence = 100 * np.max(score)

print("Predicted Disease:", predicted_class)
print("Confidence: {:.2f}%".format(confidence))

# Display image
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title(f"Prediction: {predicted_class}\nConfidence: {confidence:.2f}%")
plt.axis("off")
plt.show()